In [5]:
import yaml
import sqlglot
import sqlglot.expressions as exp

In [4]:
import sys
import os
from pathlib import Path
import yaml

project_root = Path(os.getcwd()).parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.core.parser import HiveScriptParser
from src.transformers.optimized_pyspark_transformer import OptimizedPySparkTransformer
from src.jinja.environment import render_template
from src.paths import *


In [6]:
script_name = "com_t_mhbos_m_client"
datalake_type_subfolder = 'dml'
datalake_layer_subfolder = script_name.split("_")[0]

# sql_file_path = PROJECT_ROOT / "samples" / "input" / "ddl" / "raw" / f"{script_name}.sql"
sql_file_path = DATALAKE_SCRIPT_DIR / datalake_type_subfolder / datalake_layer_subfolder / f"{script_name}.sql"
output_file_path = PROJECT_ROOT / "samples" / "converted" / datalake_type_subfolder / datalake_layer_subfolder / f"{script_name}.py"
variable_path = PROJECT_ROOT / "configs" / "rules" / "variable.yaml"


with open(sql_file_path, 'r') as f:
    ddl_content = f.read()
    ast = sqlglot.parse(ddl_content, read="hive")

variable_mappings = yaml.load(open(variable_path, 'r'), Loader=yaml.FullLoader)

'analyze table ${com_schema}.t_mhbos_m_client partition (etl_dt = '${batch_date}') compute statistics' contains unsupported syntax. Falling back to parsing as a 'Command'.


In [13]:
ast[-1]

Semicolon(_comments=[
    4.1 Drop all temporary tables,
    drop table if exists ${com_schema}.temp_t_mhbos_m_client_all;,
    drop table if exists ${com_schema}.temp_t_mhbos_m_client_primary_identification_type;,
    drop table if exists ${com_schema}.temp_t_mhbos_m_client_primary_identification_no;,
    drop table if exists ${com_schema}.temp_t_mhbos_m_client_secondary_identification_type;,
    drop table if exists ${com_schema}.temp_t_mhbos_m_client_secondary_identification_no;,
    drop table if exists ${com_schema}.temp_t_mhbos_m_client_identification_info;,
    drop table if exists ${com_schema}.temp_t_mhbos_m_client_dob_info;,
    drop table if exists ${com_schema}.temp_t_mhbos_m_client_gender_info;,
    drop table if exists ${com_schema}.temp_t_mhbos_m_client_customer_name_1;,
    drop table if exists ${com_schema}.temp_t_mhbos_m_client_customer_name_2;,
    drop table if exists ${com_schema}.temp_t_mhbos_m_client_race_info;,
    drop table if exists ${com_schema}.temp_t_mhbos

In [16]:
def is_comment_only(node):
    return (
        isinstance(node, exp.Semicolon)
        and not node.this
        and not node.args.get("expression")
    )

is_comment_only(ast[-2])

False

In [12]:
type(ast[-1])

sqlglot.expressions.Semicolon

In [21]:
# Drop
ast[1].this.this.this

'k2_bank_et'

In [50]:
# Create
ast[5].this.this.this

'k2_bank'

In [26]:
ast[4].this.this.this

'k2_bank'

In [18]:
type(ast[1])

sqlglot.expressions.Drop

In [93]:
def map_variable(query):
    for node in query.find_all(exp.Var):
        print(node.this)
        if node.this in variable_mappings:
            # Thay thế node 'Var' bằng một 'Identifier' mới
            node.replace(exp.Identifier(this=variable_mappings[node.this], quoted=False))

In [64]:
# def map_special_value(query):
#     # Thay thế các giá trị đặc biệt
#     for key, value in variable_mappings.items():
#         placeholder = f"${{{key}}}" # Tạo lại placeholder ví dụ: ${batch_timestamp}
#         if placeholder in query:
#             query = query.replace(placeholder, value)

In [97]:
for query in ast:
    map_variable(query)
    spark_sql_query = query.sql(dialect="spark")
    # map_special_value(spark_sql_query)
    print(spark_sql_query)

/* Purpose:    RAW-DDL-CREATE TABLE */ /* Author:     zjj */ /* Usage:      python $ETL_HOME/script/init.py raw k2_bank */ /* CreateDate: 20230907 */ /* FileType:   DDL */ /* Logs: */ /*     1.for hive 3.x on cdp 7.1.5 */ /* 1.0 drop table if exists table */ DROP TABLE IF EXISTS ${}.k2_bank
parquet
/* 1.1 create table */ CREATE TABLE ${}.k2_bank (bankid STRING COMMENT '', localbankcode STRING COMMENT '', bankname STRING COMMENT '', swiftcode STRING COMMENT '', oribankid STRING COMMENT '', approveuser BIGINT COMMENT '', approvets TIMESTAMP COMMENT '', createuser BIGINT COMMENT '', createts TIMESTAMP COMMENT '', updateuser BIGINT COMMENT '', updatets TIMESTAMP COMMENT '', recstatus STRING COMMENT '', etl_timestamp STRING COMMENT 'ETL_processing_time', etl_dt STRING COMMENT 'partition by day') COMMENT '' PARTITIONED BY (etl_dt) USING PARQUET TBLPROPERTIES ('parquet.compression'='SNAPPY', 'external.table.purge'='true')


In [53]:
# samples/converted/ddl/raw

render_template('pyspark/pyspark_basic.jinja', context={})

AttributeError: 'list' object has no attribute 'sql'